[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/41_max_pool2d_solution.ipynb)

# 🟡 Solution: MaxPool2d

Reference solution using `Tensor.unfold` to extract spatial windows, then reducing each window with `max`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
# ✅ SOLUTION

def _pair(value):
    if isinstance(value, tuple):
        return value
    return (value, value)


def my_max_pool2d(x: torch.Tensor, kernel_size, stride=None, padding=0) -> torch.Tensor:
    kH, kW = _pair(kernel_size)
    sH, sW = _pair(kernel_size if stride is None else stride)
    pH, pW = _pair(padding)

    if pH > 0 or pW > 0:
        x = F.pad(x, (pW, pW, pH, pH), value=float('-inf'))

    windows = x.unfold(2, kH, sH).unfold(3, kW, sW)  # (N, C, outH, outW, kH, kW)
    windows = windows.contiguous().view(*windows.shape[:4], kH * kW)
    return windows.max(dim=-1).values


In [ ]:
# Verify
x = torch.arange(1.0, 17.0).view(1, 1, 4, 4)
print("Output:")
print(my_max_pool2d(x, kernel_size=2))
print("Ref:")
print(F.max_pool2d(x, kernel_size=2))

x = torch.randn(2, 3, 5, 6)
print("Strided shape:", my_max_pool2d(x, kernel_size=(2, 3), stride=(1, 2), padding=(1, 0)).shape)


In [ ]:
# Run judge
from torch_judge import check
check('max_pool2d')
